# Feature Engineering
Turns `data/processed/Full_Merge_Cleaned.csv` (one row per school x 4-digit CIP certificate program) into model-ready data for CatBoost, LightGBM, logistic regression, random forest and gradient boosting.

- **Targets:** `vae_positive_local` (classification: locally adjusted year-1 earnings > 150% of the poverty line) and `earnings_1yr_local_adj` (regression, same programs).
- **Column decisions:** Tom's review doc (Part A drops, Part B exclusions), plus the review fixes: CIP as categories, text categoricals instead of pre-made one-hots, remaining price copies dropped, all MSI flags and race shares kept for fairness checks only.
- **Outputs:** `data/processed/model/model_features.csv` and `feature_dictionary.csv`. Encoding, scaling and imputation are **not** done here: they're model-specific and are fit inside each model notebook's training folds. The feature dictionary records each feature's type and suggested encoding so every model notebook treats the columns the same way.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_columns", None)

data_dir = Path("../../data")
df = pd.read_csv(data_dir / "processed" / "Full_Merge_Cleaned.csv", low_memory=False)
print("Cleaned dataset:", df.shape)

Cleaned dataset: (26678, 245)


## 0. Column groups
Used throughout the notebook and saved into the feature dictionary:
- **Targets**: `vae_positive_local` (classification), `earnings_1yr_local_adj` (regression) and `passes_vae_local` (the full test: value-added earnings >= program cost, fewer rows).
- **Not features**: IDs and dashboard columns, fairness-only columns (MSI flags and race shares), and meta columns (`label_source`, `split`, `cv_fold`).
- **Categoricals**: `cipcode` has 360 values (too many for one-hot; suggest target encoding); the rest are low-cardinality.
- **Log columns**: right-skewed money and count columns; suggest log1p before scaling for linear models.

In [2]:
TARGET_CLASS = "vae_positive_local"        # 1 = locally adjusted year-1 earnings > 150% FPL
TARGET_REG = "earnings_1yr_local_adj"      # same programs, dollar amount
TARGET_FULL_TEST = "passes_vae_local"      # value-added earnings >= program cost (fewer rows)
TARGETS = [TARGET_CLASS, TARGET_REG, TARGET_FULL_TEST]

ID_COLS = ["unitid", "opeid6", "instnm", "state", "city", "zip", "latitude", "longitude",
           "cipdesc", "xw_top_occupations"]
FAIRNESS_COLS = ["hbcu", "tribal", "annhi", "nanti", "hsi", "pbi", "aanapii",
                 "ugds_white", "ugds_black", "ugds_hisp", "ugds_asian", "ugds_aian",
                 "ugds_nhpi", "ugds_2mor"]
META_COLS = ["label_source", "split", "cv_fold"]

CAT_LOW = ["control", "iclevel", "bea_region", "highest_offering", "calendar_system",
           "locale", "locale_group", "preddeg", "cip_series", "price_reporting",
           "y1_earnings_natl_benchmark_level"]
CAT_HIGH = ["cipcode"]
CATEGORICAL = CAT_LOW + CAT_HIGH

LOG_COLS = ["program_cost", "tuitionfee_in", "costt4_a", "inexpfte", "avgfacsal", "ugds",
            "ipedscount1", "ipedscount2", "numbranch", "xw_occ_emp_total", "xw_skill_record_count",
            "etpl_natl_n_programs", "etpl_natl_n_with_outcomes", "etpl_state_n_programs",
            "etpl_state_n_with_outcomes", "etpl_prog_n_programs", "etpl_prog_tuition",
            "etpl_bench_tuition", "etpl_prog_contact_hours", "etpl_bench_contact_hours",
            "ay_books_supplies", "py_books_supplies", "py_offcampus_roomboard",
            "py_offcampus_other", "py_offcampus_family_other", "prog_supplies"]


def feature_columns(df, exclude=()):
    """Every column that isn't an ID, fairness, meta or target column."""
    drop = set(ID_COLS + FAIRNESS_COLS + META_COLS + TARGETS) | set(exclude)
    return [c for c in df.columns if c not in drop]

## 1. Drop columns 
- **Additions:** `xw_has_hot_technology` became constant after cleaning; the remaining 5-year PSEO columns (team decision); and the remaining price copies (`tuitionfee_out`, the `ay_*` tuition/fee variants, `py_tuitfee`), which correlate ~0.98 with the kept `tuitionfee_in` / `program_cost`.
- **Leakage columns** (Tom's Part B) are dropped from the model file; they stay in `Full_Merge_Cleaned.csv`. `earnings_1yr_local_adj` and `passes_vae_local` are kept as targets.

In [3]:
dropped = [
    "credlev", "creddesc", "curroper",
    "cipcode_str", "cip4", "cip_family", "cip_code", "cip_2digit", "st_fips",
    "tuitionfee_out_was_missing", "pctfloan_was_missing",
    "distanceonly", "locale_neg3_0", "preddeg_4_0", "highdeg_0_0", "has_pseo_match", "cip_match_level",
    "offers_undergrad", "enrolls_ft_undergrad", "offers_distance_courses",
    "xw_occ_mean_wage", "y1_earnings_natl_cip_family", "y5_earnings_natl_cip_family", "weeks_to_complete",
    "ay_tuition_indistrict", "ay_fee_indistrict", "ay_credithr_indistrict", "ay_pub_tuitfee_indistrict",
    "etpl_state_completion_median", "etpl_state_employment_q2_median", "etpl_state_earnings_median",
    "etpl_state_contact_hours_median", "etpl_state_tuition_median",
    "etpl_natl_completion_median", "etpl_natl_employment_q2_median", "etpl_natl_earnings_median",
    "etpl_natl_contact_hours_median", "etpl_natl_tuition_median",
    "local_earnings_index_state", "local_earnings_index_region",
    "prog_tuition_fees", "total_prog_cost",
    "adm_rate", "program_slot", "length_unit_code",
]
price_copies = ["tuitionfee_out", "py_tuitfee"] + [
    c for c in df.columns
    if c.startswith(("ay_tuition_", "ay_fee_", "ay_credithr_", "ay_pub_tuitfee_")) and c not in dropped]
additions = ["xw_has_hot_technology",
             "y5_earnings_natl_cip_series", "y5_earnings_natl_benchmark", "y5_earnings_natl_benchmark_level"] + price_copies

leakage = ["earn_mdn_1yr", "earn_mdn_5yr", "earn_mdn_hi_1yr", "earn_count_wne_1yr", "earn_count_wne_5yr",
           "y1_earnings_program", "y5_earnings_program",
           "y1_earnings_program_status", "y5_earnings_program_status",
           "earnings_1yr", "value_added_earnings_local", "earnings_vs_local_field", "etpl_prog_earnings"]

missing = [c for c in dropped + additions + leakage if c not in df.columns]
assert not missing, f"Not in the cleaned file: {missing}"

# Label source is needed for evaluation, so record it before the earnings columns go
df["label_source"] = np.select(
    [df[TARGET_CLASS].isna(), df["earn_mdn_1yr"].notna()],
    ["none", "scorecard"], default="pseo")

df = df.drop(columns=dropped + additions + leakage)
print(f"Dropped {len(dropped)} + {len(additions)} (additions) + {len(leakage)} (leakage) columns ->", df.shape)

Dropped 45 + 15 (additions) + 13 (leakage) columns -> (26678, 173)


C:\Users\andre\AppData\Local\Temp\ipykernel_32540\1567534511.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["label_source"] = np.select(


## 2. Categorical columns
- **One text column per category** instead of pre-made one-hots. Each model then encodes it its own way (CatBoost/LightGBM natively, one-hot for logistic regression and random forest), and SHAP shows one importance per category instead of splitting it across dummies.
- `locale` and `preddeg` only existed as one-hots, so they're rebuilt as text. `locale_group` collapses the 12 locale codes into city / suburb / town / rural.
- `sector` is dropped: it's `control` x `iclevel`, both kept.
- **CIP codes are categories, not numbers**: as integers a model would treat 5106 as "bigger than" 1609.
- `etpl_bench_level`, `local_index_level` and `length_unit` become yes/no flags.

In [4]:
onehots = [c for c in df.columns if c.startswith(("control_", "region_", "locale_", "preddeg_", "highdeg_"))]

locale_cols = [c for c in df.columns if c.startswith("locale_")]
locale_code = df[locale_cols].idxmax(axis=1).str.replace("locale_", "").str.replace("_0", "")
locale_names = {"11": "city_large", "12": "city_midsize", "13": "city_small",
                "21": "suburb_large", "22": "suburb_midsize", "23": "suburb_small",
                "31": "town_fringe", "32": "town_distant", "33": "town_remote",
                "41": "rural_fringe", "42": "rural_distant", "43": "rural_remote"}
df["locale"] = locale_code.map(locale_names)
df["locale_group"] = df["locale"].str.split("_").str[0]

preddeg_cols = [c for c in df.columns if c.startswith("preddeg_")]
preddeg_names = {"preddeg_1_0": "certificate", "preddeg_2_0": "associate", "preddeg_3_0": "bachelors"}
df["preddeg"] = df[preddeg_cols].idxmax(axis=1).map(preddeg_names)

df["etpl_bench_is_state"] = df["etpl_bench_level"].map({"state": 1.0, "national": 0.0})       # NaN = no benchmark
df["local_index_is_state"] = (df["local_index_level"] == "state").astype(float)
df["length_in_credit_hours"] = df["length_unit"].map({"credit_hours": 1.0, "contact_hours": 0.0})

df = df.drop(columns=onehots + ["sector", "etpl_bench_level", "local_index_level", "length_unit"])

df["cipcode"] = df["cipcode"].astype(str).str.zfill(4)
df["cip_series"] = df["cip_series"].astype(str).str.zfill(2)
df["calendar_system"] = df["calendar_system"].astype(str)

print("Dropped", len(onehots), "one-hot columns")
df[["locale", "locale_group", "preddeg", "control", "iclevel", "cipcode", "cip_series"]].describe()

Dropped 31 one-hot columns


C:\Users\andre\AppData\Local\Temp\ipykernel_32540\2151789834.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["locale"] = locale_code.map(locale_names)
C:\Users\andre\AppData\Local\Temp\ipykernel_32540\2151789834.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["locale_group"] = df["locale"].str.split("_").str[0]
C:\Users\andre\AppData\Local\Temp\ipykernel_32540\2151789834.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfo

,locale,locale_group,preddeg,control,iclevel,cipcode,cip_series
count,26678,26678,26678,26678,26678,26678,26678
unique,12,4,3,3,3,360,44
top,city_large,city,certificate,Public,2-year,1204,51
freq,5430,13295,14605,20703,14283,1261,5609


## 3. Engineered features
- **`entry_occ_wage`**: employment-weighted median wage of the occupations a program links to, **excluding postsecondary teachers (SOC 25-1xxx) and managers (11-xxxx)**. The crosswalk links most fields to those jobs, which need more than a certificate and inflate `xw_occ_median_wage` (e.g. massage therapy also links to Health Specialties Teachers at $127k).
- **`expected_vae_field`**: the field's typical value-added earnings in the program's state, `etpl_bench_earnings / local_earnings_index - 150% FPL`. Field-level, so no leakage.
- **`occ_wage_headroom`**: how far the entry-level occupation wage is above 150% FPL.
- **Cost and size ratios**: `cost_to_wage`, `tuition_to_wage`, `program_share_of_school`.
- **`has_program_cost`**: program-level cost exists for only 23% of programs.

In [5]:
FPL_150 = 23_475   # 150% of the federal poverty line, single person, 2025

def wavg(v, w):
    m = v.notna() & w.notna() & (w > 0)
    if m.any():
        return np.average(v[m], weights=w[m])
    return v.mean()

bridge = pd.read_csv(data_dir / "merges" / "CS_PSEO_IPEDS_BLS_merged_cip_soc_bridge.csv")
entry_level = bridge[~bridge["soc_code"].str.startswith(("25-1", "11-"))]
entry_wage = entry_level.groupby("cip4").apply(
    lambda g: wavg(g["occ_median_wage"], g["occ_emp"]), include_groups=False)
entry_wage.index = entry_wage.index.astype(str).str.zfill(4)

df["entry_occ_wage"] = df["cipcode"].map(entry_wage)
df["expected_vae_field"] = df["etpl_bench_earnings"] / df["local_earnings_index"] - FPL_150
df["occ_wage_headroom"] = df["entry_occ_wage"] - FPL_150
df["cost_to_wage"] = df["program_cost"] / df["entry_occ_wage"]
df["tuition_to_wage"] = df["tuitionfee_in"] / df["entry_occ_wage"]
df["program_share_of_school"] = df["ipedscount1"] / df["ugds"]
df["has_program_cost"] = df["program_cost"].notna().astype(float)

new_features = ["entry_occ_wage", "expected_vae_field", "occ_wage_headroom", "cost_to_wage",
                "tuition_to_wage", "program_share_of_school", "has_program_cost"]
print("Entry-level wage vs. all-occupation wage (median):",
      round(df["entry_occ_wage"].median()), "vs", round(df["xw_occ_median_wage"].median()))
df[new_features].describe().T.round(2)

Entry-level wage vs. all-occupation wage (median): 64784 vs 73092


C:\Users\andre\AppData\Local\Temp\ipykernel_32540\1808690267.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["entry_occ_wage"] = df["cipcode"].map(entry_wage)
C:\Users\andre\AppData\Local\Temp\ipykernel_32540\1808690267.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["expected_vae_field"] = df["etpl_bench_earnings"] / df["local_earnings_index"] - FPL_150
C:\Users\andre\AppData\Local\Temp\ipykernel_32540\1808690267.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `f

,count,mean,std,min,25%,50%,75%,max
entry_occ_wage,25394.0,69867.97,22869.75,35656.11,53772.83,64783.87,77560.55,267164.54
expected_vae_field,22263.0,11032.59,9542.26,-11157.76,5296.58,9386.84,15253.65,94460.16
occ_wage_headroom,25394.0,46392.97,22869.75,12181.11,30297.83,41308.87,54085.55,243689.54
cost_to_wage,6065.0,0.25,0.20,0.01,0.08,0.23,0.36,2.95
tuition_to_wage,25394.0,0.12,0.12,0.01,0.04,0.07,0.14,1.10
program_share_of_school,26678.0,0.06,0.21,0.00,0.00,0.00,0.02,10.75
has_program_cost,26678.0,0.23,0.42,0.00,0.00,0.00,0.00,1.00


## 4. Yes/no columns as numbers
Boolean columns (some with blanks, e.g. `xw_has_in_demand`) become 1 / 0 / NaN, so every non-categorical feature is numeric.

In [6]:
def to_numeric_flag(s):
    return s.map({True: 1.0, False: 0.0, "True": 1.0, "False": 0.0})

skip = CATEGORICAL + ID_COLS + TARGETS + ["label_source"]
bool_like = [c for c in df.columns
             if c not in skip and df[c].dropna().nunique() > 0
             and (df[c].dtype == bool or set(df[c].dropna().astype(str).unique()) <= {"True", "False"})]
for c in bool_like:
    df[c] = to_numeric_flag(df[c].astype(object))

df[TARGET_CLASS] = df[TARGET_CLASS].astype("Int64")
df[TARGET_FULL_TEST] = df[TARGET_FULL_TEST].astype("Int64")
print("Converted", len(bool_like), "yes/no columns")

Converted 23 yes/no columns


## 5. Train / test split and CV folds (grouped by school)
A school's programs share a lot, so splits are **by school** (`unitid`): no school appears in both train and test. Stratified on the target so pass rates match.
- `split`: `train` (80% of labeled rows), `test` (20%, held out; don't tune on it), `score_only` (no label; these get scored by the final model).
- `cv_fold`: 0-4 for training rows (5-fold grouped CV), -1 otherwise. Everyone on the team should use these columns so results are comparable.

In [7]:
labeled = df.index[df[TARGET_CLASS].notna()]
y_lab = df.loc[labeled, TARGET_CLASS].astype(int)
groups = df.loc[labeled, "unitid"]

outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_pos, test_pos = next(outer.split(labeled, y_lab, groups))

df["split"] = "score_only"
df.loc[labeled[train_pos], "split"] = "train"
df.loc[labeled[test_pos], "split"] = "test"

train_idx = labeled[train_pos]
inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
df["cv_fold"] = -1
for fold, (_, val_pos) in enumerate(inner.split(train_idx, y_lab.loc[train_idx], groups.loc[train_idx])):
    df.loc[train_idx[val_pos], "cv_fold"] = fold

df.groupby("split").agg(programs=("unitid", "size"), schools=("unitid", "nunique"),
                        pass_rate=(TARGET_CLASS, "mean")).round(3)

C:\Users\andre\AppData\Local\Temp\ipykernel_32540\1021758677.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["split"] = "score_only"
C:\Users\andre\AppData\Local\Temp\ipykernel_32540\1021758677.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["cv_fold"] = -1


,programs,schools,pass_rate
split,,,
score_only,19654,2051,<NA>
test,1405,437,0.784
train,5619,1642,0.783


## 6. Assemble and save
Column order: IDs, fairness-only, meta (`label_source`, `split`, `cv_fold`), targets, then features. The feature dictionary records each feature's score part, source, type, encoding and missingness.

In [8]:
features = feature_columns(df)
df = df[ID_COLS + FAIRNESS_COLS + META_COLS + TARGETS + features]

ELIGIBILITY = ("is_short_term_cert_provider", "is_sub_baccalaureate", "offers_cert_under_12wks", "offers_cert_12to52wks",
               "etpl_prog_wp_length_ok", "etpl_prog_likely_wp", "etpl_natl_share_wp_length", "etpl_state_share_wp_length")
LENGTH = ("py_acad_year_hours", "py_acad_year_weeks", "prog_length", "months_to_complete", "hours_per_month",
          "length_in_credit_hours", "etpl_bench_contact_hours", "etpl_prog_contact_hours")

def score_part(c):
    if c in ELIGIBILITY or c in LENGTH:
        return "eligibility / length"
    if "completion" in c or c.startswith(("c150", "ret_")):
        return "completion"
    if "employment" in c:
        return "placement"
    if c.startswith("xw_") or c in ("entry_occ_wage", "occ_wage_headroom", "has_soc_match"):
        return "demand"
    if c in ("cipcode", "cip_series", "expected_vae_field", "local_earnings_index", "local_index_is_state",
             "etpl_bench_earnings", "etpl_bench_is_state") or c.startswith("y1_earnings_natl"):
        return "earnings"
    if any(k in c for k in ("cost", "tuit", "fee", "supplies", "roomboard", "offcampus", "pctpell", "pctfloan")):
        return "cost / aid"
    if c in ("bea_region", "locale", "locale_group"):
        return "geography"
    return "institution"

def source(c):
    if c.startswith("etpl_") or c in ("expected_vae_field", "local_earnings_index", "local_index_is_state"):
        return "ETPL"
    if c.startswith("y1_earnings_natl") or c == "has_institution_cip_match":
        return "PSEO"
    if c.startswith("xw_") or c in ("entry_occ_wage", "occ_wage_headroom", "has_soc_match"):
        return "BLS / O*NET (CIP-SOC crosswalk)"
    if c in ("iclevel", "highest_offering", "calendar_system", "price_reporting", "ipeds_prog_match", "program_cost",
             "has_program_cost", "cost_to_wage", "length_in_credit_hours", "control", "bea_region")             or c.startswith(("offers_", "py_", "prog_", "months_", "hours_", "cost_per", "is_", "provides_",
                             "degree_", "enrolls_", "has_career", "ay_books")):
        return "IPEDS"
    return "College Scorecard"

def suggested_encoding(c):
    if c in CAT_HIGH:
        return "category: native (CatBoost/LightGBM), cross-fitted target encoding (others)"
    if c in CAT_LOW:
        return "category: native (CatBoost/LightGBM/HistGB), one-hot (logistic regression, random forest)"
    if c in LOG_COLS:
        return "numeric: log1p + scale (linear models); median fill + missing flag where NaN isn't supported"
    return "numeric: scale (linear models); median fill + missing flag where NaN isn't supported"

engineered = new_features + ["locale", "locale_group", "preddeg", "etpl_bench_is_state",
                             "local_index_is_state", "length_in_credit_hours"]
dictionary = pd.DataFrame({
    "feature": features,
    "score_part": [score_part(c) for c in features],
    "source": [source(c) for c in features],
    "type": ["categorical" if c in CATEGORICAL else "numeric" for c in features],
    "log_transform": [c in LOG_COLS for c in features],
    "suggested_encoding": [suggested_encoding(c) for c in features],
    "pct_missing": [round(df[c].isna().mean() * 100, 1) for c in features],
    "engineered": [c in engineered for c in features],
})
dictionary["note"] = np.where(dictionary["feature"] == "y1_earnings_natl_benchmark",
                              "Possible mild self-leak (PSEO field average includes the program's own PSEO earnings). Ablation: removing it changed HistGB CV ROC-AUC by -0.001, so kept", "")

out_dir = data_dir / "processed" / "model"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "model_features.csv", index=False)
dictionary.to_csv(out_dir / "feature_dictionary.csv", index=False)

print("Saved model_features.csv:", df.shape, f"({len(features)} features)")
print(dictionary["score_part"].value_counts().to_string())

Saved model_features.csv: (26678, 153) (123 features)
score_part
institution             48
cost / aid              23
eligibility / length    16
demand                  12
earnings                10
completion               8
geography                3
placement                3


## 7. Checks

In [9]:
check = pd.read_csv(out_dir / "model_features.csv", low_memory=False, dtype={c: str for c in CATEGORICAL})
features = feature_columns(check)

assert len(check) == 26_678
assert not check.duplicated(["unitid", "cipcode"]).any()

not_features = set(leakage + dropped + additions + ID_COLS + FAIRNESS_COLS + ["state", "sector"])
assert not not_features & set(features), not_features & set(features)
assert not set(onehots) & set(features)
assert not any(c.startswith("y5_") for c in features)

non_numeric = [c for c in features if c not in CATEGORICAL and not pd.api.types.is_numeric_dtype(check[c])]
assert not non_numeric, non_numeric

assert check.loc[check["split"] == "score_only", TARGET_CLASS].isna().all()
assert check.loc[check["split"] != "score_only", TARGET_CLASS].notna().all()

train_schools = set(check.loc[check["split"] == "train", "unitid"])
test_schools = set(check.loc[check["split"] == "test", "unitid"])
assert not train_schools & test_schools, "A school is in both train and test"

rates = check.groupby("split")[TARGET_CLASS].mean()
assert abs(rates["train"] - rates["test"]) < 0.02, rates

for k in range(5):
    fold_schools = set(check.loc[check["cv_fold"] == k, "unitid"])
    other_schools = set(check.loc[(check["cv_fold"] >= 0) & (check["cv_fold"] != k), "unitid"])
    assert not fold_schools & other_schools

print("All checks passed:", len(features), "features,", f"train/test pass rate {rates['train']:.3f} / {rates['test']:.3f}")

All checks passed: 123 features, train/test pass rate 0.783 / 0.784


### Single-feature leakage check
ROC-AUC of each numeric feature on its own (training rows). A single feature above 0.90 would suggest it's carrying the answer.

In [10]:
from sklearn.metrics import roc_auc_score

train = check[check["split"] == "train"]
y_train = train[TARGET_CLASS].astype(int)

single_auc = {}
for c in features:
    if c in CATEGORICAL:
        continue
    m = train[c].notna()
    if m.sum() >= 200 and train.loc[m, c].nunique() > 1 and y_train[m].nunique() == 2:
        auc = roc_auc_score(y_train[m], train.loc[m, c])
        single_auc[c] = max(auc, 1 - auc)

single_auc = pd.Series(single_auc).sort_values(ascending=False)
print("Features above 0.90:", list(single_auc[single_auc > 0.90].index))
single_auc.head(10).round(3).to_frame("single_feature_auc")

Features above 0.90: []


,single_feature_auc
y1_earnings_natl_benchmark,0.887
y1_earnings_natl_cip_series,0.841
tuition_to_wage,0.839
expected_vae_field,0.797
occ_wage_headroom,0.797
entry_occ_wage,0.797
xw_skill_record_count,0.796
program_share_of_school,0.794
xw_unique_software_skills,0.792
ugds,0.780


## Notes for model building
- **Use `split` and `cv_fold`** from `model_features.csv` so everyone's results are comparable; keep `test` untouched until the final model.
- **Preprocessing belongs in each model notebook** (encoding, scaling, missing-value filling), fit inside a `Pipeline` on training folds only. Read the column groups from `feature_dictionary.csv` instead of re-typing lists (snippet below).
- **Suggested preprocessing per model** (see `suggested_encoding`): CatBoost and LightGBM need none (native NaN and categoricals; CatBoost wants categoricals as strings). HistGradientBoosting: native NaN, ordinal-encode categoricals, target-encode `cipcode`. Random forest: median fill + missing flags, one-hot, target-encode `cipcode`. Logistic regression: median fill + missing flags, `log1p` on `log_transform` columns, `StandardScaler`, one-hot (`drop='first'`), target-encode `cipcode`.
- **Class imbalance (78% pass / 22% fail)**: handled with class weights, not resampling. Class weights skew predicted probabilities, so **calibrate** (e.g. `CalibratedClassifierCV`, isotonic) before using the probability as the 40% earnings part of the score.
- **Report results by group**: `control` (the labeled set is 39% for-profit vs 16% overall), `label_source` (Scorecard vs PSEO-only labels), and the fairness columns.
- **`state` is left out** of the first model (Tom's question 3); location comes through `local_earnings_index` and `bea_region`. Try adding it later and compare.
- **Leakage checks done here**: no single feature exceeds 0.90 ROC-AUC alone (highest: `y1_earnings_natl_benchmark`, 0.887), and removing that feature changed an untuned HistGradientBoosting baseline by -0.001 ROC-AUC (0.947 vs 0.948), so it's kept.

**Loading the data in a model notebook:**
```python
import pandas as pd
from pathlib import Path

model_dir = Path("../../data/processed/model") 
dictionary = pd.read_csv(model_dir / "feature_dictionary.csv")
features = dictionary["feature"].tolist()
categorical = dictionary.loc[dictionary["type"] == "categorical", "feature"].tolist()
log_cols = dictionary.loc[dictionary["log_transform"], "feature"].tolist()

df = pd.read_csv(model_dir / "model_features.csv", low_memory=False, dtype={c: str for c in categorical})
train = df[df["split"] == "train"]
X_train, y_train = train[features], train["vae_positive_local"].astype(int)
cv_folds = train["cv_fold"]          # PredefinedSplit(cv_folds) for grouped CV
```